# Final no-anchor training-health confirmation（6/20）

这是进入正式 Stage 5 前的最后一次独立健康确认。固定 `policy_max_norm=5`、`retry=3`，从全新 policy/optimizer/scheduler 开始；不恢复旧 Step 12 或 clipping A/B 训练状态，不使用短期 Reward/IC 选择参数，也不会自动创建正式训练 run。

### 第 0 格：检查最终候选配置与安全开关

这一格只构造配置并显示 F/D/E/L、输入文件、32-batch 预算和健康门槛，不加载真实数据，通常几秒完成。首次运行保持 `RUN_MODE='new'`；确认后把 `RUN_REAL_FINAL_HEALTH=False` 改成 `True`。

In [2]:
import json, math, sys, torch
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'factor_gfn').is_dir())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from factor_gfn.gfn import (ExhaustiveRegistry, GFNTrainer, LogZInitializationHealthConfig, NoAnchorCalibrationConfig, RealRewardDataPaths, RealRewardProvider, TrainingConfig, build_formal_stage5_no_anchor_6_20_config, build_log_z_initialization_health, build_real_reward_data_context)
from factor_gfn.gfn.diagnostic_support import PhaseTrackingRewardProvider, configure_registry_once, progress_heartbeat, run_training_with_progress

RUN_REAL_FINAL_HEALTH = False
RUN_MODE = 'new'  # 中断后改为 'resume'，然后从第0格顺序运行
DEVICE = 'cuda:0'
LOGICAL_BATCHES = 32
SOURCE_DIAGNOSTIC_ROOT = root / 'runs' / 'complexity_diagnostic_6_20' / 'manual_diagnostic_6_20_seed42'
SOURCE_REGISTRY = SOURCE_DIAGNOSTIC_ROOT / 'exhaustive_registry.sqlite3'
TARGETED_ARTIFACT = root / 'runs' / 'targeted_calibration_6_20' / 'targeted_logz_n17_n18_seed42' / 'targeted_log_z_engineering_initialization.json'
RUN_ROOT = root / 'runs' / 'final_health_confirmation_6_20' / 'final_health_seed42'
LATEST_CHECKPOINT = RUN_ROOT / 'latest_no_anchor.pt'
training = TrainingConfig(batch_size=8, learning_rate=1e-4, log_z_learning_rate=1e-2, max_steps=LOGICAL_BATCHES + 2, model_gradient_clip_norm=5.0, log_z_gradient_clip_norm=5.0, seed=42)
base_config = build_formal_stage5_no_anchor_6_20_config(training=training)
config = replace(base_config, complexity=replace(base_config.complexity, exact_node_retry_budget=3), calibration=NoAnchorCalibrationConfig(enabled=True, target_node_counts=(17, 18)))
health_config = LogZInitializationHealthConfig()
strata = config.resolved_strata()
print({'enabled': RUN_REAL_FINAL_HEALTH, 'run_mode': RUN_MODE, 'device': DEVICE, 'run_root': str(RUN_ROOT), 'logical_batches': LOGICAL_BATCHES, 'F': strata.feasible_node_counts, 'D': strata.discovery_node_counts, 'E': strata.exact_normalizer_node_counts, 'L': strata.learned_normalizer_node_counts, 'policy_max_norm': 5.0, 'retry_budget': 3, 'batch_size': 8, 'policy_lr': 1e-4, 'logZ_lr': 1e-2, 'logZ_max_norm': 5.0, 'health_config': asdict(health_config), 'config_fingerprint': config.fingerprint()}, flush=True)
print('预计32个batch约30–40分钟；末尾确定性恢复再约2–4分钟。每个batch逐项输出，内部每20秒heartbeat。', flush=True)

{'enabled': True, 'run_mode': 'new', 'device': 'cuda:0', 'run_root': 'D:\\实习\\Gflownet因子挖掘\\runs\\final_health_confirmation_6_20\\final_health_seed42', 'logical_batches': 32, 'F': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20), 'D': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20), 'E': (1, 2), 'L': (3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20), 'policy_max_norm': 5.0, 'retry_budget': 3, 'batch_size': 8, 'policy_lr': 0.0001, 'logZ_lr': 0.01, 'logZ_max_norm': 5.0, 'health_config': {'minimum_valid_trajectories_per_N': 7, 'minimum_successful_gradient_exposures_per_N': 7, 'early_exposure_count': 3, 'late_exposure_count': 3, 'initial_abs_delta_mean_review_threshold': 5.0, 'late_abs_delta_mean_review_threshold': 2.5, 'log_z_net_change_review_threshold': 0.75, 'directional_drift_fraction_threshold': 0.8, 'minimum_log_z_step_for_direction': 0.0001}, 'config_fingerprint': 'a56cb11bf4da0b782cdfdb9570b07c6705cffa1272730c23e69261a

### 第 1 格：建立或恢复独立最终 health run

这一格加载真实 training-only 数据、检查行业中性化，重新完成只读 registry 等价证明。`new` 模式创建全新 Trainer，导入 historical median 后只用工程 artifact 覆盖 N=17/18；`resume` 只恢复本 run 的新 schema checkpoint。不会重新执行 exhaustive Reward 或 calibration。通常数分钟，超过20秒持续输出心跳。

In [3]:
if not RUN_REAL_FINAL_HEALTH: raise RuntimeError('安全停止：确认第0格后，将 RUN_REAL_FINAL_HEALTH=True')
if RUN_MODE not in {'new','resume'}: raise ValueError("RUN_MODE只能是'new'或'resume'")
if not DEVICE.startswith('cuda:') or not torch.cuda.is_available(): raise RuntimeError('最终health必须显式使用CUDA，不回落CPU')
for required in (SOURCE_REGISTRY, TARGETED_ARTIFACT, SOURCE_DIAGNOSTIC_ROOT / 'diagnostic_summary.json', SOURCE_DIAGNOSTIC_ROOT / 'diagnostic_context.json'):
    if not required.is_file(): raise FileNotFoundError(required)
device=torch.device(DEVICE); torch.cuda.set_device(device); torch.cuda.reset_peak_memory_stats(device)
if RUN_MODE=='new': RUN_ROOT.mkdir(parents=True, exist_ok=False)
elif not RUN_ROOT.is_dir(): raise FileNotFoundError('resume要求既有同名最终health目录')
print('[provider] loading training-only context; heartbeat every 20s', flush=True)
with progress_heartbeat('final health provider load', interval_seconds=20.0):
    context=build_real_reward_data_context(paths=RealRewardDataPaths())
    base_provider=RealRewardProvider(context, config.reward)
    provider=PhaseTrackingRewardProvider(base_provider, audit_path=RUN_ROOT/'reward_phase_audit.jsonl')
manifest=provider.manifest()
assert manifest['data_scope']=='training_only' and manifest['validation_oos_loaded'] is False
assert manifest['industry_neutralization']['enabled'] and base_provider.reward_config.candidate_industry_neutralization
registry=ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
trainer=GFNTrainer(config, provider, device=device)
with progress_heartbeat('final health registry equivalence proof', interval_seconds=20.0): configure_registry_once(trainer, registry)
if RUN_MODE=='resume':
    if not LATEST_CHECKPOINT.is_file(): raise FileNotFoundError(LATEST_CHECKPOINT)
    trainer.load_checkpoint(LATEST_CHECKPOINT)
    historical=trainer.historical_log_z_initialization; targeted=trainer.targeted_log_z_initialization
    initial_manifest=json.loads((RUN_ROOT/'initialization_manifest.json').read_text(encoding='utf-8'))
    initial_log_z_by_N={int(k):float(v) for k,v in initial_manifest['initial_log_z_by_N'].items()}
    print(f'[resume] logical={trainer.step}, successful={trainer.optimizer_step}', flush=True)
else:
    with progress_heartbeat('final health historical initialization proof', interval_seconds=20.0): historical=trainer.initialize_verified_historical_log_z(SOURCE_DIAGNOSTIC_ROOT)
    targeted=trainer.initialize_verified_targeted_log_z(TARGETED_ARTIFACT)
    assert targeted.initialization_status=='high_variance_engineering_estimate' and targeted.strict_stability_check=='failed'
    initial_log_z_by_N={**{n:float(trainer.tb_loss.exact_tb_log_z_by_node_count[n-1]) for n in trainer.resolved_exhaustive_node_counts}, **{n:float(trainer.tb_loss.log_z_by_node_count[n-1]) for n in trainer.resolved_learned_node_counts}}
    initial_manifest={'schema':'factor_gfn.final_health_initialization.v1','config_fingerprint':config.fingerprint(),'provider_fingerprint':provider.fingerprint(),'historical_provenance_fingerprint':historical.provenance_fingerprint,'targeted_provenance_fingerprint':targeted.provenance_fingerprint,'targeted_status':targeted.initialization_status,'strict_stability_check':targeted.strict_stability_check,'initial_log_z_by_N':initial_log_z_by_N}
    (RUN_ROOT/'initialization_manifest.json').write_text(json.dumps(initial_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    trainer.save_checkpoint(LATEST_CHECKPOINT)
if targeted is None or historical is None: raise RuntimeError('初始化provenance不完整')
assert set(initial_log_z_by_N)==set(trainer.resolved_feasible_node_counts)
print('[initialization] exact N=1/2 and learned N=3..20 ready; N17/N18=',{n:initial_log_z_by_N[n] for n in (17,18)},flush=True)
print('[initialization] no old model/optimizer/scheduler state was imported in new mode',flush=True)

[provider] loading training-only context; heartbeat every 20s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[registry-proof] verified_once={1: 6, 2: 636}
[initialization] exact N=1/2 and learned N=3..20 ready; N17/N18= {17: 46.30188751220703, 18: 46.4272575378418}
[initialization] no old model/optimizer/scheduler state was imported in new mode


### 第 2 格：运行32个 logical batch

这是主要长计算格。每个 batch 开始和结束都打印 logical/successful/skipped、loss、TB RMS、retry、耗时和 ETA；batch 内每20秒 heartbeat；每个 batch 原子保存 checkpoint。中断后设 `RUN_MODE='resume'`，重启内核并从第0格顺序运行。预计约30–40分钟。

In [4]:
training_path=RUN_ROOT/'training_health.jsonl'; trajectory_path=RUN_ROOT/'trajectory_tb_diagnostics.jsonl'
remaining=max(0,LOGICAL_BATCHES-trainer.step)
if remaining:
    print(f'[training] remaining logical batches={remaining}; current successful={trainer.optimizer_step}',flush=True)
    with provider.phase('final_health_discovery'):
        run_training_with_progress(trainer,logical_batches=remaining,checkpoint_path=LATEST_CHECKPOINT,checkpoint_every=1,training_audit_path=training_path,trajectory_audit_path=trajectory_path)
else: print('[training] 32-batch budget already complete in checkpoint',flush=True)
print(f'[training] complete logical={trainer.step}, successful={trainer.optimizer_step}, skipped={trainer.step-trainer.optimizer_step}',flush=True)

[training] remaining logical batches=32; current successful=0
[training] starting invocation_batch=1/32, logical_batch=1, trainer_step=0, successful_updates=0
[training logical_batch=1] still running; elapsed=20.0s
[training logical_batch=1] still running; elapsed=40.1s
[training logical_batch=1] still running; elapsed=60.2s
[training] invocation_batch=1/32, logical_batch=1, successful_updates=1, skipped=False, loss=6.496564202875464, tb_rms=2.5488358524776493, requested={1: 0, 2: 0, 3: 0, 4: 0, 5: 1, 6: 1, 7: 0, 8: 0, 9: 0, 10: 1, 11: 0, 12: 0, 13: 0, 14: 1, 15: 1, 16: 1, 17: 0, 18: 0, 19: 1, 20: 1}, retries={1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0, 16: 0, 17: 0, 18: 0, 19: 0, 20: 0}, elapsed=75.8s, eta=2348.4s
[training] checkpoint=D:\实习\Gflownet因子挖掘\runs\final_health_confirmation_6_20\final_health_seed42\latest_no_anchor.pt
[training] starting invocation_batch=2/32, logical_batch=2, trainer_step=1, successful_updates=1
[training

### 第 3 格：生成 per-N 最终健康报告

这一格不再训练，通常几秒完成。它按 logical batch 去重审计，分别保留每个 N 的 initialization/pre-update、early、late TB delta，以及 initial/current/net-change logZ、valid和successful exposure；同时汇总梯度、裁剪、实际更新、retry、吞吐和显存。证据不足或异常只报告，不自动重置logZ或修改参数。

In [5]:
raw_training=[json.loads(x) for x in training_path.read_text(encoding='utf-8').splitlines() if x.strip()]
training_rows=list({int(r['logical_batch']):r for r in raw_training}.values()); training_rows.sort(key=lambda r:int(r['logical_batch']))
raw_trajectory=[json.loads(x) for x in trajectory_path.read_text(encoding='utf-8').splitlines() if x.strip()]
trajectory_rows=list({(int(r['logical_batch']),int(r['candidate_index_in_batch'])):r for r in raw_trajectory}.values()); trajectory_rows.sort(key=lambda r:(int(r['logical_batch']),int(r['candidate_index_in_batch'])))
current_log_z_by_N={**{n:float(trainer.tb_loss.exact_tb_log_z_by_node_count[n-1]) for n in trainer.resolved_exhaustive_node_counts}, **{n:float(trainer.tb_loss.log_z_by_node_count[n-1]) for n in trainer.resolved_learned_node_counts}}
health=build_log_z_initialization_health(trajectory_rows,training_rows,node_counts=trainer.resolved_feasible_node_counts,learned_node_counts=trainer.resolved_learned_node_counts,initial_log_z_by_N=initial_log_z_by_N,current_log_z_by_N=current_log_z_by_N,config=health_config)
health_rows=[]
for n,item in health.per_N.items():
    phases=item['tb_delta_by_phase']; health_rows.append({'N':n,'kind':item['normalizer_kind'],'status':item['status'],'valid_trajectories':item['valid_trajectory_count'],'successful_gradient_exposures':item['successful_gradient_exposure_count'],'initial_delta_mean':phases['initialization_pre_update']['mean'],'initial_delta_std':phases['initialization_pre_update']['std'],'early_delta_mean':phases['early']['mean'],'early_delta_std':phases['early']['std'],'late_delta_mean':phases['late']['mean'],'late_delta_std':phases['late']['std'],'initial_log_z':item['initial_log_z'],'current_log_z':item['current_log_z'],'net_change_log_z':item['net_change_log_z'],'directional_update_fraction':item['directional_log_z_update_fraction']})
health_frame=pd.DataFrame(health_rows).set_index('N'); display(health_frame)
health_frame.to_csv(RUN_ROOT/'final_health_by_N.csv'); pd.DataFrame(health.enriched_trajectory_rows).to_csv(RUN_ROOT/'trajectory_tb_health_phases.csv',index=False)
training_frame=pd.json_normalize(training_rows); training_frame.to_csv(RUN_ROOT/'training_health.csv',index=False)
display(training_frame[['logical_batch','optimizer_step','loss','tb_delta_rms','model_gradient_norm_before_clip','model_gradient_clip_coefficient','model_parameter_update_norm','policy_entropy_mean','batch_wall_seconds','cuda_peak_memory_bytes']])
successful=[r for r in training_rows if r['successful_optimizer_update']]
nonfinite=sum(not math.isfinite(float(r[k])) for r in successful for k in ('loss','tb_delta_rms','model_gradient_norm_before_clip','model_parameter_update_norm','policy_entropy_mean'))
counters={'requested_count_by_N':dict(trainer.requested_count_by_N),'valid_count_by_N':dict(trainer.valid_count_by_N),'sampled_attempt_count_by_N':dict(trainer.sampled_attempt_count_by_N),'retry_exhausted_count_by_N':dict(trainer.retry_exhausted_count_by_N),'successful_update_count_by_N':dict(trainer.successful_update_count_by_N)}
evidence_status='insufficient_evidence' if health.insufficient_evidence_node_counts else ('review_required' if health.targeted_recalibration_node_counts or nonfinite else 'candidate_pass')
report={'schema':'factor_gfn.final_no_anchor_health_confirmation.v1','evidence_status':evidence_status,'final_stage5_decision':'manual_review_required','all_learned_initializations_usable':health.all_learned_initializations_usable,'review_node_counts':health.targeted_recalibration_node_counts,'insufficient_evidence_node_counts':health.insufficient_evidence_node_counts,'initialization_health_by_N':health.per_N,'initial_log_z_by_N':initial_log_z_by_N,'current_log_z_by_N':current_log_z_by_N,'logical_batches':len(training_rows),'successful_updates':len(successful),'skipped_updates':len(training_rows)-len(successful),'skip_rate':(len(training_rows)-len(successful))/len(training_rows),'nonfinite_health_values':nonfinite,'policy_gradient_clip_fraction':float(np.mean([r['model_gradient_clip_coefficient']<1.0 for r in successful])),'policy_gradient_clip_coefficient_median':float(np.median([r['model_gradient_clip_coefficient'] for r in successful])),'model_parameter_update_norm_median':float(np.median([r['model_parameter_update_norm'] for r in successful])),'tb_rms_first4_mean':float(np.mean([r['tb_delta_rms'] for r in successful[:4]])),'tb_rms_last4_mean':float(np.mean([r['tb_delta_rms'] for r in successful[-4:]])),'cuda_peak_memory_bytes':int(torch.cuda.max_memory_allocated(device)),**counters}
(RUN_ROOT/'final_health_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('[health]',{k:report[k] for k in ('evidence_status','review_node_counts','insufficient_evidence_node_counts','logical_batches','successful_updates','skipped_updates','skip_rate','nonfinite_health_values','tb_rms_first4_mean','tb_rms_last4_mean')},flush=True)
print('[health] 该状态只供人工确认；不会自动进入Stage 5。',flush=True)

,kind,status,valid_trajectories,successful_gradient_exposures,initial_delta_mean,initial_delta_std,early_delta_mean,early_delta_std,late_delta_mean,late_delta_std,initial_log_z,current_log_z,net_change_log_z,directional_update_fraction
N,,,,,,,,,,,,,,
1,exact_fixed,fixed_exact_diagnostic,13,13,0.295095,0.0,0.321040,0.016721,0.303028,0.020341,-1.766961,-1.766961,0.000000,NaN
2,exact_fixed,fixed_exact_diagnostic,12,12,1.146679,0.0,0.764726,1.131562,0.858784,1.509528,2.849463,2.849463,0.000000,NaN
3,learned,usable,12,12,-0.021956,0.0,0.365982,0.432264,0.376396,1.644252,5.751882,5.707985,-0.043897,0.916667
4,learned,usable,13,13,0.911178,0.0,0.591171,0.701125,0.698202,0.771544,8.557557,8.528000,-0.029557,0.916667
5,learned,usable,13,13,0.868243,0.0,0.083016,1.984513,-3.171836,4.410684,12.044566,12.025394,-0.019172,0.692308
6,learned,usable,13,13,2.350506,0.0,1.492713,2.452398,1.250786,0.609455,15.843430,15.812887,-0.030542,0.692308
7,learned,usable,12,12,-3.368806,0.0,0.407047,2.091194,3.727066,1.708660,19.031773,19.041296,0.009523,0.666667
8,learned,usable,13,13,1.728363,0.0,0.664007,1.239730,-1.262508,3.091465,23.181034,23.170433,-0.010601,0.538462
9,learned,usable,13,13,-4.657044,0.0,0.276570,0.293050,-0.431921,3.173614,26.251713,26.264919,0.013206,0.700000


,logical_batch,optimizer_step,loss,tb_delta_rms,model_gradient_norm_before_clip,model_gradient_clip_coefficient,model_parameter_update_norm,policy_entropy_mean,batch_wall_seconds,cuda_peak_memory_bytes
0,1,1,6.496564,2.548836,114.573688,0.043640,0.096706,4.015339,75.750782,137513984
1,2,1,NaN,NaN,NaN,NaN,NaN,NaN,62.239683,137513984
2,3,2,5.932804,2.435735,107.529741,0.046499,0.072115,3.713206,34.589674,137513984
3,4,3,43.678029,6.608936,407.511428,0.012270,0.056076,4.175977,55.432197,137513984
4,5,4,34.191143,5.847319,290.319180,0.017222,0.048989,3.929970,53.701521,137890816
5,6,5,9.499501,3.082126,133.573110,0.037433,0.045401,3.894282,35.335452,137890816
6,7,6,19.187000,4.380297,248.963187,0.020083,0.041589,4.058375,117.407438,169212928
7,8,7,4.553615,2.133920,107.357882,0.046573,0.035641,4.048126,32.654759,169212928
8,9,8,13.081763,3.616872,181.720397,0.027515,0.033447,3.898911,42.834183,169212928
9,10,9,16.152029,4.018959,243.837866,0.020505,0.031817,3.989934,62.198977,169212928


[health] {'evidence_status': 'candidate_pass', 'review_node_counts': (), 'insufficient_evidence_node_counts': (), 'logical_batches': 32, 'successful_updates': 31, 'skipped_updates': 1, 'skip_rate': 0.03125, 'nonfinite_health_values': 0, 'tb_rms_first4_mean': 4.360206343341333, 'tb_rms_last4_mean': 4.654227424741547}
[health] 该状态只供人工确认；不会自动进入Stage 5。


### 第 4 格：验证 checkpoint 确定性恢复

最后保存冻结的32-batch状态，然后从同一 checkpoint 分别执行 source/resume 各一个 batch，逐值比较目标N、轨迹、loss、logZ、scheduler和模型。两个真实batch通常约2–4分钟，各自每20秒 heartbeat；它们只用于确定性验证，不写入前面的32-batch健康审计。

In [6]:
checkpoint=RUN_ROOT/'checkpoint_before_determinism.pt'; trainer.save_checkpoint(checkpoint)
print('[determinism] source batch starting; heartbeat every20s',flush=True)
with provider.phase('determinism_source'):
    with progress_heartbeat('final health determinism source',interval_seconds=20.0): expected=trainer.train_step()
expected_diag=trainer.last_discovery_trajectory_diagnostics; expected_model={k:v.detach().clone() for k,v in trainer.model.state_dict().items()}; expected_log_z=trainer.tb_loss.log_z_by_node_count.detach().clone(); expected_scheduler=trainer.complexity_scheduler.state_dict()
resumed=GFNTrainer(config,provider,device=device); configure_registry_once(resumed,registry); resumed.load_checkpoint(checkpoint)
print('[determinism] resume batch starting; heartbeat every20s',flush=True)
with provider.phase('determinism_resume'):
    with progress_heartbeat('final health determinism resume',interval_seconds=20.0): actual=resumed.train_step()
assert actual==expected and resumed.last_discovery_trajectory_diagnostics==expected_diag
assert resumed.complexity_scheduler.state_dict()==expected_scheduler and torch.equal(resumed.tb_loss.log_z_by_node_count,expected_log_z)
assert all(torch.equal(resumed.model.state_dict()[k],v) for k,v in expected_model.items())
report['checkpoint_resume_exact']=True; (RUN_ROOT/'final_health_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
registry.close()
print('FINAL_NO_ANCHOR_HEALTH_CONFIRMATION_COMPLETE',flush=True)
print('把结果目录交给Codex分析：',RUN_ROOT,flush=True)

[determinism] source batch starting; heartbeat every20s
[final health determinism source] still running; elapsed=20.0s
[final health determinism source] still running; elapsed=40.1s
[registry-proof] verified_once={1: 6, 2: 636}
[determinism] resume batch starting; heartbeat every20s
[final health determinism resume] still running; elapsed=20.0s
FINAL_NO_ANCHOR_HEALTH_CONFIRMATION_COMPLETE
把结果目录交给Codex分析： D:\实习\Gflownet因子挖掘\runs\final_health_confirmation_6_20\final_health_seed42
